# Continual learning using predictive coding

This notebook builds on [`1_supervised_learning_pc.ipynb`](predictive-coding/1_supervised_learning_pc.ipynb) from the Bogacz Group's predictive coding library. There, a predictive coding network was trained on a single task (MNIST). Here we ask what happens when the same network has to learn a *sequence* of tasks, one after the other, without revisiting old data — the **continual learning** setting.

Neural networks trained with backpropagation typically suffer from **catastrophic forgetting**: learning a new task overwrites the weights that supported previous tasks, so performance on old tasks collapses. Brains do not appear to forget this way, and there is evidence that energy-based learning schemes such as predictive coding degrade more gracefully (see Song et al., *Nature Neuroscience* 2024, on prospective configuration).

We test this on the classic **Permuted MNIST** benchmark:

1. build a sequence of tasks, each a pixel-permuted version of MNIST,
2. train a predictive coding network on the tasks sequentially,
3. train a backpropagation network of the same architecture the same way,
4. compare how much each model forgets.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ha0wan9/NMA-NeuroAI-Online-Learning-Project/blob/main/playground/continual_learning_pc.ipynb)

In [ ]:
# make the predictive_coding library importable
# - on google colab: clone the Bogacz-Group repository
# - locally: the library lives in the `predictive-coding/` git submodule next to this notebook
try:
  import google.colab
  !git clone https://github.com/Bogacz-Group/PredictiveCoding.git
  ! cp -r PredictiveCoding/predictive_coding predictive_coding
except ImportError:
  import os, sys
  sys.path.insert(0, os.path.join(os.getcwd(), 'predictive-coding'))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import predictive_coding as pc

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'using {device}')

### The permuted MNIST benchmark

Each task in permuted MNIST is the full MNIST dataset with one fixed, random shuffling of the 784 pixel positions. The classes and the amount of information in each image are identical across tasks, but the input statistics are completely different, so a network cannot solve a new task by reusing the input features of an old one — it has to adapt its weights. This makes the benchmark a clean probe of forgetting: any drop in accuracy on task 0 while training on task 3 is caused by the weight updates for task 3.

We keep the identity permutation as task 0, so task 0 is exactly the MNIST task from the supervised learning notebook.

In [ ]:
# load data
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: torch.flatten(x))])
mnist_train = datasets.MNIST('./data', download=True, train=True, transform=transform)
mnist_test = datasets.MNIST('./data', download=True, train=False, transform=transform)

# one fixed pixel permutation per task; task 0 keeps the original pixel order
n_tasks = 5
generator = torch.Generator().manual_seed(0)
permutations = [torch.arange(28*28)] + [torch.randperm(28*28, generator=generator) for _ in range(n_tasks - 1)]

class PermutedMNIST(torch.utils.data.Dataset):
    """MNIST with a fixed permutation applied to the flattened pixels."""
    def __init__(self, dataset, permutation):
        self.dataset = dataset
        self.permutation = permutation

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]
        return image[self.permutation], label

train_tasks = [PermutedMNIST(mnist_train, p) for p in permutations]
test_tasks = [PermutedMNIST(mnist_test, p) for p in permutations]

batch_size = 500
print(f'{n_tasks} tasks with {len(mnist_train)} train and {len(mnist_test)} test images each')

In [ ]:
# a fixed, colorblind-safe color per task, used consistently in every figure below
TASK_COLORS = ['#2a78d6', '#008300', '#e87ba4', '#eda100', '#1baf7a']
PC_COLOR, BP_COLOR = '#2a78d6', '#eb6834'

# the same digit under each task's permutation
image, label = mnist_train[0]
fig, axes = plt.subplots(1, n_tasks, figsize=(2 * n_tasks, 2.4))
for task_id, (ax, permutation) in enumerate(zip(axes, permutations)):
    ax.imshow(image[permutation].reshape(28, 28), cmap='gray_r')
    ax.set_title(f'task {task_id}', color=TASK_COLORS[task_id])
    ax.axis('off')
fig.suptitle(f'the same digit (a {label}) under each task permutation')
plt.show()

### Defining the model

The model is identical to the one in the supervised learning notebook: a fully connected network with `pc.PCLayer()` modules holding the latent variables of each hidden layer. A `PCLayer()` stores the activity of a layer of latent variables (`pclayer._x`) and the energy of that layer (`pclayer._energy`), computed as `0.5 * (inputs['mu'] - inputs['x'])**2` where `inputs['mu']` is the prediction arriving at the layer.

We wrap the construction in a function so we can create fresh models for each experiment.

In [ ]:
input_size = 28*28  # 28x28 images
hidden_size = 256
output_size = 10    # 10 classes
activation_fn = nn.ReLU
loss_fn = lambda output, _target: 0.5 * (output - _target).pow(2).sum() # holds the error of the output layer

def make_pc_model():
    model = nn.Sequential(
        nn.Linear(input_size, hidden_size),
        pc.PCLayer(),
        activation_fn(),
        nn.Linear(hidden_size, hidden_size),
        pc.PCLayer(),
        activation_fn(),
        nn.Linear(hidden_size, output_size)
    )
    model.train()   # set the model to training mode
    return model.to(device)

pc_model = make_pc_model()
pc_model

### Defining a model trainer

As in the supervised learning notebook, a `pc.PCTrainer()` orchestrates inference and learning: for each batch it first runs `T` iterations of inference, updating the latent states `x` to reduce the total energy of the network, and then updates the parameters `p` at the last inference step.

This inference phase is what we expect to matter for continual learning. Because the latent activities settle into a configuration consistent with the target *before* the weights are updated, the weight updates can be smaller and better targeted than a backpropagation step, which should disturb previously learned tasks less (Song et al. 2024 call this prospective configuration).

In [ ]:
# number of inference iterations where the latent states x are updated
T = 20

# options for the update of the latent state x
optimizer_x_fn = optim.SGD
optimizer_x_kwargs = {'lr': 0.01}

# options for the update of the parameters p
update_p_at = 'last'                # update parameters p at the last inference iteration
optimizer_p_fn = optim.Adam
optimizer_p_kwargs = {'lr': 0.001}

trainer = pc.PCTrainer(pc_model,
    T = T,
    optimizer_x_fn = optimizer_x_fn,
    optimizer_x_kwargs = optimizer_x_kwargs,
    update_p_at = update_p_at,
    optimizer_p_fn = optimizer_p_fn,
    optimizer_p_kwargs = optimizer_p_kwargs,
)

In [ ]:
# get classification accuracy of the model on a dataset
def test(model, dataset, batch_size=1000):
    model.eval()
    test_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    correct = 0
    total = 0
    for data, label in test_loader:
        data, label = data.to(device), label.to(device)
        pred = model(data)
        _, predicted = torch.max(pred, -1)
        total += label.size(0)
        correct += (predicted == label).sum().item()
    model.train()
    return round(correct / total, 4)

### Training sequentially on the tasks

The continual learning protocol: train on task 0 for a few epochs, then task 1, and so on. Data from earlier tasks is never revisited — no replay buffer, no regularisation towards old weights, no task labels at test time. After every epoch we evaluate the model on the test set of *every* task, which gives us the full history of how performance on each task evolves while the model learns the others.

The training harness below is model-agnostic: it takes a function that trains on one batch, so we can reuse it unchanged for the backpropagation baseline.

In [ ]:
epochs_per_task = 3

def train_continually(train_batch_fn, model):
    """Train sequentially on each task, evaluating on all tasks after every epoch.

    Returns an array of shape (n_tasks * epochs_per_task + 1, n_tasks) where row k
    holds the test accuracy on every task after k epochs of training (row 0 is the
    untrained model).
    """
    accuracies = [[test(model, task) for task in test_tasks]]
    for task_id, task in enumerate(train_tasks):
        loader = torch.utils.data.DataLoader(task, batch_size=batch_size, shuffle=True)
        for epoch in range(epochs_per_task):
            desc = f'task {task_id}, epoch {epoch + 1}/{epochs_per_task}'
            for data, label in tqdm(loader, desc=desc):
                data, label = data.to(device), label.to(device)
                # convert labels to one-hot encoding
                target = F.one_hot(label, num_classes=output_size).float()
                train_batch_fn(data, target)
            accuracies.append([test(model, task) for task in test_tasks])
    return np.array(accuracies)

In [ ]:
# train the predictive coding model
def pc_train_batch(data, target):
    trainer.train_on_batch(
        inputs=data,
        loss_fn=loss_fn,
        loss_fn_kwargs={'_target': target}
    )

pc_accuracies = train_continually(pc_train_batch, pc_model)
print(f'final accuracy per task: {pc_accuracies[-1]}')

### A backpropagation baseline

To see whether predictive coding forgets less, we need a reference point: the same architecture (without the `PCLayer`s), the same output loss, the same parameter optimizer, trained with plain backpropagation on the same task sequence.

In [ ]:
def make_bp_model():
    model = nn.Sequential(
        nn.Linear(input_size, hidden_size),
        activation_fn(),
        nn.Linear(hidden_size, hidden_size),
        activation_fn(),
        nn.Linear(hidden_size, output_size)
    )
    model.train()
    return model.to(device)

bp_model = make_bp_model()
bp_optimizer = optimizer_p_fn(bp_model.parameters(), **optimizer_p_kwargs)

def bp_train_batch(data, target):
    bp_optimizer.zero_grad()
    loss = loss_fn(bp_model(data), target)
    loss.backward()
    bp_optimizer.step()

bp_accuracies = train_continually(bp_train_batch, bp_model)
print(f'final accuracy per task: {bp_accuracies[-1]}')

### How much does each model forget?

First, the full picture: the accuracy of each task over the course of training, from the moment that task starts being trained. Vertical lines mark task boundaries. A model that does not forget would keep every curve flat after its task ends; catastrophic forgetting shows up as curves dropping as soon as the next task begins.

In [ ]:
def plot_task_accuracies(ax, accuracies, title):
    epochs = np.arange(accuracies.shape[0])
    for task_id in range(n_tasks):
        start = task_id * epochs_per_task  # epoch at which this task starts training
        ax.plot(epochs[start:], accuracies[start:, task_id],
                color=TASK_COLORS[task_id], lw=2, label=f'task {task_id}')
    for task_id in range(1, n_tasks):
        ax.axvline(task_id * epochs_per_task, color='0.85', lw=1, zorder=0)
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.set_ylim(0, 1)
    ax.spines[['top', 'right']].set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
plot_task_accuracies(axes[0], pc_accuracies, 'predictive coding')
plot_task_accuracies(axes[1], bp_accuracies, 'backpropagation')
axes[0].set_ylabel('test accuracy')
axes[1].legend(loc='lower left', frameon=False)
fig.suptitle('accuracy on each task while training on the tasks sequentially')
plt.tight_layout()
plt.show()

Two summary numbers make the comparison concrete:

- **Average accuracy over seen tasks** after each epoch — how well the model does on everything it has been taught so far.
- **Average forgetting** — for each task (except the last), the accuracy it reached right after being trained minus its accuracy at the very end of training. Zero means no forgetting.

In [ ]:
def average_over_seen_tasks(accuracies):
    """Mean accuracy over the tasks trained so far, after each epoch."""
    averages = np.full(accuracies.shape[0], np.nan)
    for row in range(1, accuracies.shape[0]):
        n_seen = min((row - 1) // epochs_per_task + 1, n_tasks)
        averages[row] = accuracies[row, :n_seen].mean()
    return averages

def average_forgetting(accuracies):
    """Mean drop from each task's accuracy right after its training to its final accuracy."""
    end_of_task = accuracies[np.arange(1, n_tasks + 1) * epochs_per_task, np.arange(n_tasks)]
    final = accuracies[-1]
    return (end_of_task - final)[:-1].mean()  # the last task has had no chance to be forgotten

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(average_over_seen_tasks(pc_accuracies), color=PC_COLOR, lw=2, label='predictive coding')
ax.plot(average_over_seen_tasks(bp_accuracies), color=BP_COLOR, lw=2, label='backpropagation')
for task_id in range(1, n_tasks):
    ax.axvline(task_id * epochs_per_task, color='0.85', lw=1, zorder=0)
ax.set_xlabel('epoch')
ax.set_ylabel('average accuracy over seen tasks')
ax.set_ylim(0, 1)
ax.legend(loc='lower left', frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

print(f'average forgetting - predictive coding: {average_forgetting(pc_accuracies):.3f}, '
      f'backpropagation: {average_forgetting(bp_accuracies):.3f}')

### What to expect, and what to try next

Both models learn each new task to high accuracy, and both forget older tasks to some degree — neither is a full solution to continual learning. The interesting comparison is *how much* they forget: with this setup the predictive coding network typically retains noticeably more accuracy on earlier tasks than the backpropagation baseline, consistent with the continual learning results reported for prospective configuration by Song et al. (2024). Because the latent activities settle towards a configuration that already explains the target before any weight changes, the weight updates interfere less with what the network learned before.

Some experiments to try:

- **Inference duration `T`**: with fewer inference steps (e.g. `T = 5`) the latent states settle less before the weights update, and predictive coding behaves more like backpropagation. Does forgetting increase?
- **Incremental PC**: set `update_p_at = 'all'` to update the weights at every inference step ([iPC](https://arxiv.org/abs/2212.00720)). How does this change the stability–plasticity trade-off?
- **Online learning**: set `batch_size = 1` (and reduce `epochs_per_task`) for fully online continual learning, the regime most relevant to biological learning.
- **Longer task sequences**: increase `n_tasks` to 10. Does the gap between the two models grow or shrink?
- **Learning rate of the latent states**: `optimizer_x_kwargs['lr']` controls how far inference relaxes the network. How sensitive are the results to it?

### References

- Song, Y., Millidge, B., Salvatori, T., et al. [Inferring neural activity before plasticity as a foundation for learning beyond backpropagation](https://doi.org/10.1038/s41593-023-01514-1). *Nature Neuroscience* 27, 348–358 (2024).
- Goodfellow, I. J., et al. [An empirical investigation of catastrophic forgetting in gradient-based neural networks](https://arxiv.org/abs/1312.6211) (2013) — the paper that popularised permuted MNIST.
- Bogacz Group, [PredictiveCoding library](https://github.com/Bogacz-Group/PredictiveCoding) — the library and tutorial series this notebook builds on.